# Systematic Review
**References:**
* ❌ too old - https://github.com/chandraveshchaudhari/systematic-reviewpy
* ❌ agentic AI - https://github.com/PouriaRouzrokh/LatteReview
* ✅ Uses PubMed API - https://github.com/gijswobben/pymed

**TO-DO**
* ✅ Clean this notebook up
* Run the query on other databases as well
* QC the results from each database, starting with pubmed.
* Once QC'd, merge the results across each database result and process.

**REFERENCES THAT CAUGHT MY EYE**
* https://pubmed.ncbi.nlm.nih.gov/33202965/
* https://pubmed.ncbi.nlm.nih.gov/38389433/

# Environment Setup

In [16]:
# Import all required packages
from datetime import datetime
from dotenv import load_dotenv
from pathlib import Path
from pymed import PubMed
import json
import os
import pandas as pd
import re
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

# Load environment variables
load_dotenv()

True

# Specify Search Strategy

Create a GraphQL query in plain text <br>
Hack: I used the query builder on https://pubmed.ncbi.nlm.nih.gov/advanced/ to create this




In [2]:
# This is the bottom-up query that guided the keywords I wanted.
query = '"(geospatial analysis) AND (parkinson* disease) NOT ((motor) OR (global burden))"' # results = 6

In [ ]:
# These are big-to-small queries.
#query = "((geospatial analysis) OR (geographic analysis)) AND (parkinson* disease) AND (environmental atmospheric)"
#query = "(parkinson* disease[title]) AND ((geospatial analysis) OR (pollution)) NOT ((motor) OR (genetic) OR (neurologic) OR (nicotine))" # results = 149
#query = "(parkinson* disease[title]) AND ((geospatial analysis) OR (pollution)) NOT ((motor) OR (genetic) OR (neurologic) OR (nicotine) OR (smoking))" # results = 129

# Query PubMed

In [3]:
# Create a PubMed object that GraphQL can use to query
# Note that the parameters are not required but kindly requested by PubMed Central
# https://www.ncbi.nlm.nih.gov/pmc/tools/developers/
pubmed = PubMed(tool="MyTool", email=os.getenv("PUBMED_EMAIL"))

In [4]:
# Execute the query against the API
# Convert to list so the results can be iterated multiple times (not exhausted)
results = list(pubmed.query(query, max_results=100))

# Initial Query Results

### Preview query results as a DataFrame

This cell converts `results` (a list of PubMed article objects) into a pandas DataFrame for quick inspection. It shows the first 10 rows and creates `df_results` for further exploration. If `results` is missing/empty, the preview cell prints a message instead.

In [7]:
# Preview `results` as a pandas DataFrame
# This cell converts the `results` list (PubMed article objects) into a DataFrame
# and displays a concise preview (first N rows).

if 'results' not in globals() or not results:
    print("`results` is not defined or empty. Run the query cell above to populate `results`.")
else:
    def _article_to_flat_dict(article):
        # Try to leverage article.toJSON() when available
        try:
            raw = article.toJSON()
            if isinstance(raw, str):
                return json.loads(raw)
            if isinstance(raw, dict):
                return raw
        except Exception:
            pass

        # Fallback: extract common attributes safely
        return {
            "pubmed_id": getattr(article, "pubmed_id", None),
            "title": getattr(article, "title", None),
            "publication_date": str(getattr(article, "publication_date", "") or ""),
            "keywords": getattr(article, "keywords", None),
            "abstract": getattr(article, "abstract", None),
        }

    df_results = pd.json_normalize([_article_to_flat_dict(a) for a in results])


In [17]:
# Preview the columns labels as a list before displaying the DataFrame
print(df_results.columns.tolist())

['abstract', 'authors', 'conclusions', 'copyrights', 'doi', 'journal', 'keywords', 'methods', 'publication_date', 'pubmed_id', 'results', 'title', 'xml']


In [18]:
# Pick nice default preview columns (if present)
preview_cols = [c for c in ["title", "abstract", "publication_date", "keywords"] if c in df_results.columns]

# Make output readable in the notebook
pd.set_option('display.max_colwidth', 200)
n = 10
print(f"Showing first {min(n, len(df_results))} of {len(df_results)} results (variable: df_results)")
display(df_results[preview_cols].head(n))

Showing first 6 of 6 results (variable: df_results)


,title,abstract,publication_date,keywords
0,Wastewater-borne markers of neurodegenerative disease: β-methylamino-L-alanine and aminomethylphosphonic acid.,"Exposure to toxic organic chemicals such as β-methylamino-L-alanine (BMAA) and glyphosate has been associated with neurodegenerative diseases (NDDs), including amyotrophic lateral sclerosis (ALS),...",2025-03-09,"[Environmental toxicants, Liquid chromatography tandem mass spectroscopy, Neurodegeneration, Prevalence, Wastewater-based epidemiology, β-Methylamino-L-alanine]"
1,Literature review and meta-analysis of environmental toxins associated with increased risk of Parkinson's disease.,"Parkinson's disease (PD) is a neurodegenerative disorder and leading cause of death worldwide, whose pathogenesis has been linked to toxic environmental exposures. We used the Preferred Reporting ...",2024-04-30,"[Case-control studies, Cohort studies, Environmental factors, Geographic distribution, Parkinson's disease]"
2,Traffic-related air pollution and Parkinson's disease in central California.,Prior studies suggested that air pollution exposure may increase the risk of Parkinson's Disease (PD). We investigated the long-term impacts of traffic-related and multiple sources of particulate ...,2023-10-20,"[Air pollution, Case-control study, Long-term exposure, Parkinson's disease]"
3,Geospatial Analysis of Persons with Movement Disorders Living in Underserved Regions.,Movement disorders persons from underserved areas have increased barriers to access tertiary care. There is currently limited data on the geographic and demographic profile of movement disorders p...,2021-09-14,"[Movement Disorders, geography, spatial analysis, underserved]"
4,Geospatial analysis of individual-based Parkinson's disease data supports a link with air pollution: A case-control study.,"The etiology of Parkinson's disease (PD) remains unknown. To approach the issue of PD's risk factors from a new perspective, we hypothesized that coupling the geographic distribution of PD with sp...",2021-01-22,"[Air pollution, Environment, Epidemiology, Parkinson's disease, Prevalence, Spatial dependence]"
5,Geospatial Analysis of Environmental Atmospheric Risk Factors in Neurodegenerative Diseases: A Systematic Review.,"Despite the vast evidence on the environmental influence in neurodegenerative diseases, those considering a geospatial approach are scarce. We conducted a systematic review to identify studies con...",2020-11-19,"[environment, epidemiology, geospatial, neurodegenerative, systematic review]"


# Refining the Query
I'll use NLP to identify the keywords to expand the initial query on.

In [ ]:
# Combine fields into single text per document
# NOTE: make checks robust against numpy arrays / lists / pandas NA (avoid ambiguous truth values)

def _to_text(row, cols=('title', 'abstract', 'keywords')):
    parts = []
    for c in cols:
        # row might be a dict (to_dict orient='records') or a pandas Series
        if isinstance(row, dict):
            val = row.get(c, '')
        else:
            # Series-like
            val = row.get(c, '') if c in row else ''

        # Handle common NA / empty cases safely (avoid pd.isna() directly in an if)
        if val is None:
            val = ''
        elif isinstance(val, float) and np.isnan(val):
            val = ''
        elif isinstance(val, (list, tuple)):
            # join list-like values
            if len(val) == 0:
                val = ''
            else:
                val = ' '.join(map(str, val))
        elif isinstance(val, (np.ndarray,)):
            # convert numpy arrays to list then join
            if val.size == 0:
                val = ''
            else:
                val = ' '.join(map(str, val.tolist()))
        else:
            # keep whatever string representation
            val = '' if (isinstance(val, float) and np.isnan(val)) else str(val)

        parts.append(val)

    # Only return joined string from non-empty parts
    return ' '.join([p for p in parts if p])

# Small helper to normalize candidate terms when comparing

def _normalize(s):
    s = re.sub(r"[^a-z0-9\s]"," ", s.lower())
    s = re.sub(r"\s+"," ", s).strip()
    return s


In [29]:
# Combine relevant text columns into a single corpus (safe now)
try:
    corpus = [_to_text(r) for r in df_results.to_dict(orient='records')]
    print(f"Built corpus with {len(corpus)} documents")
    if corpus:
        print("Sample (first document, first 300 chars):\n", corpus[0][:300])
except Exception as e:
    # Provide a helpful debugging hint rather than failing silently
    raise RuntimeError("Failed to build corpus from df_results — check the data types in your text columns") from e


Built corpus with 6 documents
Sample (first document, first 300 chars):
 Wastewater-borne markers of neurodegenerative disease: β-methylamino-L-alanine and aminomethylphosphonic acid. Exposure to toxic organic chemicals such as β-methylamino-L-alanine (BMAA) and glyphosate has been associated with neurodegenerative diseases (NDDs), including amyotrophic lateral sclerosis


In [30]:
# Parse out current query words/phrases to avoid suggesting exact duplicates
existing_query_terms = set([_normalize(t) for t in re.findall(r"[A-Za-z*]+(?:\s+[A-Za-z*]+)*", query)])

# NOTE: _to_text now handles array/list/NA safely — this avoids ambiguous truth-value errors


In [31]:
# TF-IDF over 1..3 grams
tfidf = TfidfVectorizer(ngram_range=(1,3), stop_words='english', max_df=0.9)
X = tfidf.fit_transform(corpus)
feature_names = tfidf.get_feature_names_out()

# Sum TF-IDF across rows to get corpus-level importance
scores = np.asarray(X.sum(axis=0)).ravel()

tfidf_df = pd.DataFrame({'term': feature_names, 'tfidf': scores})

In [32]:
# Raw frequency counts for same n-grams
cv = CountVectorizer(ngram_range=(1,3), stop_words='english')
Y = cv.fit_transform(corpus)
freqs = np.asarray(Y.sum(axis=0)).ravel()
cv_terms = cv.get_feature_names_out()

freq_df = pd.DataFrame({'term': cv_terms, 'count': freqs})

In [33]:
# Merge the two recommendation signals
cand = tfidf_df.merge(freq_df, on='term', how='outer').fillna(0)
# Add a combined score that balances TF-IDF and frequency
cand['score'] = cand['tfidf'] * 0.7 + (cand['count'] / (cand['count'].max() + 1e-9)) * 0.3

# Add ngram length (for grouping / filtering)
cand['ngram_len'] = cand['term'].str.count(' ') + 1

# Normalize terms for filtering against the query
cand['term_norm'] = cand['term'].apply(_normalize)
# Filter out terms included in the query and short stopwords
cand = cand[~cand['term_norm'].isin(existing_query_terms)]
# Exclude single characters and pure numbers
cand = cand[cand['term'].str.len() > 2]
cand = cand[~cand['term'].str.match(r"^\d+$")]

In [34]:
# Show top candidates for each n-gram size
top_n = 30
results_unified = cand.sort_values('score', ascending=False).head(top_n)

print(f"Found {len(cand)} candidate terms; showing top {len(results_unified)} overall")

# Display grouped results with helpful columns
display_cols = ['term', 'ngram_len', 'count', 'tfidf', 'score']
print('\nTop candidates (combined score):')
display(results_unified[display_cols].reset_index(drop=True))

# Also show top single-word and multi-word candidates separately
for n in (1,2,3):
    section = cand[cand['ngram_len']==n].sort_values('score', ascending=False).head(12)
    if not section.empty:
        print(f"\nTop {len(section)} ngram_len={n} candidates:")
        display(section[display_cols].reset_index(drop=True))

# Provide a simple function to return a list of recommended terms to extend the query
def suggest_expansions(n=20, min_count=1):
    s = cand[cand['count'] >= min_count].sort_values('score', ascending=False)
    return list(s['term'].head(n))

print('\nExample: call suggest_expansions() to get a list of suggested terms for expanding your query')


Found 2307 candidate terms; showing top 30 overall

Top candidates (combined score):


,term,ngram_len,count,tfidf,score
0,environmental,1,15,0.370008,0.473291
1,prevalence,1,11,0.338147,0.393846
2,pollution,1,11,0.322496,0.382890
3,neurodegenerative,1,9,0.329804,0.359434
4,underserved,1,9,0.312956,0.347641
5,air,1,10,0.290759,0.346388
6,air pollution,2,10,0.290759,0.346388
7,exposure,1,10,0.247010,0.315764
8,parkinson,1,11,0.218119,0.309826
9,wastewater,1,7,0.251165,0.275816



Top 12 ngram_len=1 candidates:


,term,ngram_len,count,tfidf,score
0,environmental,1,15,0.370008,0.473291
1,prevalence,1,11,0.338147,0.393846
2,pollution,1,11,0.322496,0.382890
3,neurodegenerative,1,9,0.329804,0.359434
4,underserved,1,9,0.312956,0.347641
5,air,1,10,0.290759,0.346388
6,exposure,1,10,0.247010,0.315764
7,parkinson,1,11,0.218119,0.309826
8,wastewater,1,7,0.251165,0.275816
9,exposures,1,7,0.231728,0.262210



Top 12 ngram_len=2 candidates:


,term,ngram_len,count,tfidf,score
0,air pollution,2,10,0.290759,0.346388
1,movement disorders,2,6,0.208638,0.231761
2,traffic related,2,4,0.183369,0.185501
3,neurodegenerative diseases,2,4,0.181746,0.184365
4,environmental atmospheric,2,3,0.185757,0.172887
5,systematic review,2,3,0.185757,0.172887
6,case control,2,5,0.140970,0.170107
7,uf nfind,2,4,0.139092,0.154507
8,odds ratio,2,4,0.119029,0.140463
9,long term,2,3,0.137527,0.139126



Top 12 ngram_len=3 candidates:


,term,ngram_len,count,tfidf,score
0,parkinson disease pd,3,4,0.091506,0.121197
1,factors neurodegenerative diseases,3,2,0.123838,0.115258
2,environmental atmospheric risk,3,2,0.123838,0.115258
3,atmospheric risk factors,3,2,0.123838,0.115258
4,risk factors neurodegenerative,3,2,0.123838,0.115258
5,case control study,3,3,0.102236,0.114422
6,traffic related air,3,2,0.091685,0.092751
7,related air pollution,3,2,0.091685,0.092751
8,t3 vs t1,3,2,0.091685,0.092751
9,toxic organic chemicals,3,2,0.071761,0.078804



Example: call suggest_expansions() to get a list of suggested terms for expanding your query


### How the automatic suggestions work

The cell above bundles text from `title`, `abstract`, and `keywords` into a single document per article and runs two straightforward, explainable techniques:

- TF-IDF (1..3-grams) to find terms that are especially discriminative in the corpus
- Raw n-gram frequency counts (1..3-grams) to surface common phrases

These signals are combined into a single score (weighted) and shown by n-gram length. Use the helper function `suggest_expansions(n=20, min_count=1)` to retrieve the top-n recommended terms.

Tuning tips:
- Increase `min_count` to require a phrase appear in more documents (reduces rare/noisy terms).
- Reduce `max_df` / adjust stop-word handling in the TF-IDF vectorizer to remove overly-common terms.
- If you'd like, I can add an additional extractor (e.g., RAKE/YAKE or spaCy noun-chunk extraction) — tell me which direction you prefer.

In [ ]:
# This is the final query.
query = "(parkinson* disease[title]) AND ((geospatial analysis) OR (pollution)) NOT ((motor) OR (genetic) OR (neurologic) OR (nicotine) OR (smoking) OR (halitosis) OR (disease-like) OR (treatment[title]) OR (neuroprotective[title]) OR (maternal) OR (preventative) OR (therapy))" # results = 86

# Export Query Results

In [7]:
# Config
MAX_RESULTS = 100
OUT_DIR = Path("pubmed_results")
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [8]:
# Ensure `results` is available and not an exhausted iterator
try:
    has_items = hasattr(results, "__len__") and len(results) > 0
except NameError:
    has_items = False

if not has_items:
    print("`results` is empty or undefined — running the query to fetch items")
    # Re-run the query and store as list so it can be reused
    results = list(pubmed.query(query, max_results=MAX_RESULTS))

In [9]:
# Timestamped filename to avoid accidental overwrites
timestamp = datetime.utcnow().strftime('%Y%m%dT%H%M%SZ')
OUT = OUT_DIR / f"results-{timestamp}.ndjson"
QUERY_FILE = OUT_DIR / f"query-{timestamp}.txt"

In [10]:
def article_to_dict(article):
    # Prefer the object's own JSON if available
    try:
        raw = article.toJSON()
        if isinstance(raw, str):
            return json.loads(raw)
        if isinstance(raw, dict):
            return raw
    except Exception:
        pass

    # Fallback: extract common fields safely
    return {
        "pubmed_id": getattr(article, "pubmed_id", None),
        "title": getattr(article, "title", None),
        "keywords": [k for k in (getattr(article, "keywords", []) or []) if k],
        "publication_date": str(getattr(article, "publication_date", "") or ""),
        "abstract": getattr(article, "abstract", None),
    }

count = 0
with OUT.open("w", encoding="utf-8") as fh:
    for a in results:
        try:
            obj = article_to_dict(a)
            fh.write(json.dumps(obj, ensure_ascii=False))
            fh.write("\n")
            count += 1
        except Exception as e:
            # log and continue
            print(f"Failed to write article {getattr(a, 'pubmed_id', '<unknown>')}: {e}")

In [11]:
# Write the query metadata and query string to a timestamped text file
try:
    with QUERY_FILE.open("w", encoding="utf-8") as qf:
        qf.write(f"timestamp: {timestamp}\n")
        qf.write(f"max_results: {MAX_RESULTS}\n")
        qf.write("query:\n")
        qf.write(query)
except Exception as e:
    print(f"Failed to write query file: {e}")

print(f"Wrote {count} records to {OUT.resolve()}")
print(f"Wrote query file to {QUERY_FILE.resolve()}")

Wrote 86 records to /mnt/c/Users/ReginaChua/Desktop/sysrev/pubmed_results/results-20251030T083228Z.ndjson
Wrote query file to /mnt/c/Users/ReginaChua/Desktop/sysrev/pubmed_results/query-20251030T083228Z.txt


# Get a list of articles
Convert the latest NDJSON to a GitHub-friendly CSV (saved in project root)

In [13]:
# Configure paths - NDJSON in pubmed_results/, CSV in project root
OUT_DIR = Path('pubmed_results')
csv_name = Path(f'results-{datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")}.csv')

In [14]:
# Re-use the DataFrame if it exists, otherwise load from NDJSON
if 'ndjson_df' not in globals():
    # Find latest NDJSON
    files = sorted(OUT_DIR.glob('results*.ndjson'), key=lambda p: p.stat().st_mtime, reverse=True)
    if not files:
        raise FileNotFoundError("No NDJSON files found. Run the export cell first.")
    
    latest = files[0]
    print(f"Loading from {latest.name}")
    
    # Load NDJSON
    records = []
    with latest.open('r', encoding='utf-8') as fh:
        for line in fh:
            if line.strip():
                records.append(json.loads(line))
    
    # Create DataFrame
    ndjson_df = pd.json_normalize(records)
    
    # Convert keywords to strings if present
    if 'keywords' in ndjson_df.columns:
        ndjson_df['keywords'] = ndjson_df['keywords'].apply(lambda k: ', '.join(k) if isinstance(k, (list, tuple)) else k)


In [15]:
# Save as CSV with minimal processing for GitHub readability
try:
    # Reorder columns for readability (put common fields first)
    preferred = ['pubmed_id', 'title', 'publication_date', 'keywords', 'abstract']
    cols = [c for c in preferred if c in ndjson_df.columns] + [c for c in ndjson_df.columns if c not in preferred]
    
    # Write CSV (UTF-8 encoding, no index) to project root
    ndjson_df[cols].to_csv(csv_name, index=False, encoding='utf-8')
    print(f"Saved CSV to project root: {csv_name}")
    
except Exception as e:
    print(f"Error saving CSV: {e}")

Saved CSV to project root: results-20251030T083636Z.csv
